# Planted Noisy kXOR: Block-encoding the Kikuchi Matrix

Section 4.4.2 Simulating the Kikuchi Hamiltonian

This module contains oracles to implement the block-encoding of the Kikuchi
Hamiltonian corresponding to an input k-XOR-SAT instance.

References:
    [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1)
    Section 4.4.2 for algorithm. Section 2.4 for definitions and notation.

Complete native counterpart of the Qualtran v0.7.0 notebook, revision
`bcb17bb25360ed68ce3bc084645db3e929799389`, frozen blob
`78031452acc39de315339e2962f85067473c49a9`. Copyright 2024 Google LLC,
adapted under Apache-2.0.

These examples reproduce source declarations, signatures, call graphs and QEC
estimates. Root physical execution, packed-array admission, complex amplitudes,
measurement branches and work-zero acceptance remain open in
`KIKUCHI_BLOCK_ENCODING_FILE_AUDIT.md`. Executable shared children retain
`circuit.append(op)`. The model estimates below are separate from emitted
primitive counts.


In [ ]:
from collections.abc import Callable
from typing import cast

import sympy
from IPython import display as notebook_display

from flagquantum.algorithms.kikuchi_block_encoding import KikuchiHamiltonian
from flagquantum.algorithms.kxor_instance import KXorInstance, example_kxor_instance
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(
        type(register.bitsize) is int
        and all(type(size) is int for size in register.shape_symbolic)
        for register in signature
    ):
        dot = export_signature_graph_dot(signature, name)
    else:
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " shape"] = str(register.shape_symbolic)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `KikuchiHamiltonian`
Block encoding of the Kikuchi matrix $\mathcal{K}_\ell$.

This is implemented by a sparse matrix block encoding using the adjacency matrix
and adjacency list oracles.

This assumes a default sparsity of $\bar{m}$, which is the number of unique
scopes in the instance $\mathcal{I}$.
If a better bound on sparsity is known, it can be passed in by the user.

#### Parameters
 - `inst`: kXOR instance $\mathcal{I}$.
 - `ell`: Kikuchi parameter $\ell$.
 - `entry_bitsize`: Number of bits $b$ to approximate the matrix entries (angles) to.
 - `s`: sparsity of the Kikuchi matrix, defaults to $\bar{m}$.


In [ ]:
# The Hamiltonian declaration is imported in the setup cell.


### Example Instances

In [ ]:
inst = example_kxor_instance()
ell = 8
kikuchi_matrix = KikuchiHamiltonian(inst, ell)


In [ ]:
n, m, k, c = sympy.symbols("n m k c", positive=True, integer=True)
inst = KXorInstance.symbolic(n=n, m=m, k=k)
ell = c * k
kikuchi_matrix_symb = KikuchiHamiltonian(inst, ell)


#### Graphical Signature

In [ ]:
show_signature("kikuchi_matrix", kikuchi_matrix.signature)
show_signature("kikuchi_matrix_symb", kikuchi_matrix_symb.signature)


### Call Graph

In [ ]:
kikuchi_matrix_graph = kikuchi_matrix.source_graph(
    root="KikuchiHamiltonian", sparse_encoding="SparseMatrixHermitian"
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                kikuchi_matrix_graph.root, kikuchi_matrix_graph.direct_calls
            )
        )
    )
)
kikuchi_matrix_sigma = kikuchi_matrix_graph.leaf_counts
display(kikuchi_matrix_sigma)


In [ ]:
model_categories = kikuchi_matrix.resource_gates()
symbolic_model_categories = kikuchi_matrix_symb.resource_gates()
controlled_model_categories = kikuchi_matrix.resource_gates(controlled=True)
assert model_categories.clifford == 310479
assert model_categories.cswap == 13312
assert controlled_model_categories.cswap == 13345
display(model_categories)
display(symbolic_model_categories)
display(controlled_model_categories)
